# Label-Ambiguity Ceiling — How Much of the Residual Error Is Learnable?

**Analysis notebook, not a model.** Motivated by four independent modelling results that all failed
in the same direction:

| Experiment | Outcome | What it hinted |
|---|---|---|
| Focal weighting | collapsed at γ=2 (0.3497) | up-weighting *hard* samples destroys the model → hard samples may be **unlearnable**, not informative |
| Dynamic ensemble (v1) | 0.0804 oracle headroom, none realised | errors are **shared** across models |
| Per-class thresholds | every gain inverted on test | the decision rule is not the constraint |
| Wider α grid (v3) | headroom 0.1456→0.2250, gain +0.004 | more disagreement ≠ more recoverable signal |

Every one of these is consistent with a residual error that is a property of **the data**, not of the
classifier. This notebook measures that directly.

## The argument

The dataset was deduplicated on **features + label**, so every surviving repeated feature vector is
one carrying **conflicting labels**. Any deterministic classifier is a function of $x$ — it must emit
one label per distinct feature vector — so its accuracy is bounded above by

$$\text{acc}^\star = \frac{1}{N}\sum_{\text{groups }g} \max_{c} \; n_{g,c}$$

where $n_{g,c}$ counts rows in group $g$ carrying label $c$. This is a **strict upper bound**, achieved
only by memorising the majority label of every group. It is the Bayes error under the given feature
representation.

The same construction gives a **per-class ceiling**: a class cannot be recalled on rows where it is
not the majority label of its own group.

## What this delivers for the thesis

If Shellcode's ceiling is near its achieved 0.367, then "weak rare-class recall" stops being a
limitation a panel presses on and becomes a **quantified property of CIC-UNSW-NB15-2024**. Almost no
NIDS paper reports this, and it speaks directly to the *weak recall on rare classes* gap from the
literature review.

> Runtime ~2 min, dominated by loading the CSVs.

## 1. Load and reproduce the exact pipeline state

Same dedupe, same prune, same split seed as every model notebook, so the test set is identical
row-for-row and the ceiling is directly comparable to the reported scores.

In [1]:
import numpy as np, pandas as pd, warnings
from itertools import combinations
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, recall_score
from scipy.stats import spearmanr

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS = 42

X_raw = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y_raw = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
print("raw rows: %d" % len(X_raw))

df = pd.concat([X_raw, y_raw], axis=1).drop_duplicates().reset_index(drop=True)
X76, y = df[X_raw.columns], df["Label"]
print("after dedupe on features+label: %d  (%d removed)" % (len(df), len(X_raw) - len(df)))

nun   = X76.nunique(); zero = nun[nun <= 1].index.tolist()
corr  = X76.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X44 = X76.drop(columns=list(dict.fromkeys(zero + red)))
print("features: %d -> %d after the label-free prune" % (X76.shape[1], X44.shape[1]))

# identical outer split -> identical test set
idx_all = np.arange(len(y))
idx_trf, idx_test = train_test_split(idx_all, test_size=0.2, random_state=RS, stratify=y)
print("train+val %d | test %d" % (len(idx_trf), len(idx_test)))

raw rows: 447915
after dedupe on features+label: 306173  (141742 removed)
features: 76 -> 44 after the label-free prune
train+val 244938 | test 61235


## 2. Group identical feature vectors

64-bit hashing of the feature rows. Collision probability at this scale is ~$n^2/2^{65} \approx 10^{-9}$,
negligible.

**Both feature sets are measured.** If the 76→44 prune created ambiguity that was not there before,
the prune itself injected label noise and that would be a finding about the pipeline, not the data.

In [2]:
def group_stats(Xf, y, tag):
    key = pd.util.hash_pandas_object(Xf, index=False).values
    g   = pd.DataFrame({"k": key, "y": y.values})
    cnt = g.groupby(["k", "y"]).size().rename("n").reset_index()
    gsz = cnt.groupby("k")["n"].sum()
    gmx = cnt.groupby("k")["n"].max()
    nuq = cnt.groupby("k").size()                       # distinct labels per group
    maj = cnt.loc[cnt.groupby("k")["n"].idxmax()].set_index("k")["y"]
    amb_keys  = nuq[nuq > 1].index
    amb_rows  = gsz[amb_keys].sum()
    print("--- %s ---" % tag)
    print("  distinct feature vectors : %d" % len(gsz))
    print("  ambiguous groups         : %d" % len(amb_keys))
    print("  rows in ambiguous groups : %d  (%.2f%%)" % (amb_rows, 100 * amb_rows / len(g)))
    print("  Bayes-optimal accuracy   : %.4f" % (gmx.sum() / len(g)))
    print("  -> irreducible error     : %.4f" % (1 - gmx.sum() / len(g)))
    return dict(key=key, cnt=cnt, maj=maj, nuq=nuq, amb=set(amb_keys))

S76 = group_stats(X76, y, "76 features (pre-prune)")
print()
S44 = group_stats(X44, y, "44 features (the modelling representation)")

extra = len(S44["amb"]) - len(S76["amb"])
print("\nambiguous groups created by the 76->44 prune: %d" % extra)
print("-> the prune is %s" % ("essentially label-noise-neutral" if abs(extra) < 0.02 * len(S76["amb"])
                              else "MATERIALLY changing ambiguity -- investigate"))

--- 76 features (pre-prune) ---
  distinct feature vectors : 305602
  ambiguous groups         : 497
  rows in ambiguous groups : 1068  (0.35%)
  Bayes-optimal accuracy   : 0.9981
  -> irreducible error     : 0.0019

--- 44 features (the modelling representation) ---
  distinct feature vectors : 305602
  ambiguous groups         : 497
  rows in ambiguous groups : 1068  (0.35%)
  Bayes-optimal accuracy   : 0.9981
  -> irreducible error     : 0.0019

ambiguous groups created by the 76->44 prune: 0
-> the prune is essentially label-noise-neutral


## 3. The ceiling, on the test set

The bound is evaluated where the models are scored. Two versions:

- **Global ceiling** — majority label taken over the whole dataset. The true upper bound for any
  deterministic classifier with perfect knowledge.
- **Memoriser** — majority taken over `train+val` only and applied to test. What perfect memorisation
  of the training data would actually score; rows whose feature vector never appears in training are
  unconstrained and fall back to Benign.

In [3]:
key = S44["key"]
y_arr = y.values

# --- global ceiling ---
oracle = S44["maj"].reindex(key).values
y_test = y_arr[idx_test]
ceil_pred = oracle[idx_test]
print("GLOBAL CEILING on the test set")
print("  accuracy   : %.4f" % (ceil_pred == y_test).mean())
print("  macro-F1   : %.4f" % f1_score(y_test, ceil_pred, average="macro", zero_division=0))

# --- memoriser trained on train+val ---
tr_tab = (pd.DataFrame({"k": key[idx_trf], "y": y_arr[idx_trf]})
          .groupby(["k", "y"]).size().rename("n").reset_index())
tr_maj = tr_tab.loc[tr_tab.groupby("k")["n"].idxmax()].set_index("k")["y"]
seen   = pd.Series(key[idx_test]).isin(tr_maj.index).values
mem_pred = tr_maj.reindex(key[idx_test]).fillna(0).astype(int).values
print("\nMEMORISER (majority over train+val, applied to test)")
print("  test rows whose feature vector appears in train+val: %d (%.2f%%)"
      % (seen.sum(), 100 * seen.mean()))
print("  accuracy   : %.4f" % (mem_pred == y_test).mean())
print("  macro-F1   : %.4f" % f1_score(y_test, mem_pred, average="macro", zero_division=0))
if seen.sum():
    print("  on those seen rows, train's majority label differs from the test label: %.2f%%"
          % (100 * (mem_pred[seen] != y_test[seen]).mean()))
    print("  (dedupe was on features+label, so a vector present in BOTH splits necessarily")
    print("   carries conflicting labels -- these rows are guaranteed errors for any classifier)")

GLOBAL CEILING on the test set
  accuracy   : 0.9979
  macro-F1   : 0.9881

MEMORISER (majority over train+val, applied to test)
  test rows whose feature vector appears in train+val: 192 (0.31%)
  accuracy   : 0.7345
  macro-F1   : 0.0848
  on those seen rows, train's majority label differs from the test label: 100.00%
  (dedupe was on features+label, so a vector present in BOTH splits necessarily
   carries conflicting labels -- these rows are guaranteed errors for any classifier)


## 4. Per-class ceiling vs achieved

The central table. `ceiling_R` is the highest recall physically attainable for a class: it cannot be
recovered on rows where it is not the majority label of its own feature group.

Achieved figures are the **v3 10-member ensemble** (test macro-F1 0.6329) — the configuration whose
full per-class report was produced in `two_stage_stageB_ensemble_v3.ipynb` §5.2.

In [4]:
V3_F1 = dict(zip(NAMES, [0.983, 0.455, 0.634, 0.462, 0.773, 0.708, 0.805, 0.657, 0.367, 0.486]))
V3_R  = dict(zip(NAMES, [0.968, 0.818, 0.511, 0.404, 0.696, 0.942, 0.774, 0.568, 0.368, 0.375]))

ceil_rep = classification_report(y_test, ceil_pred, labels=list(range(10)),
                                 target_names=NAMES, output_dict=True, zero_division=0)
rows = []
for i, n in enumerate(NAMES):
    cr, cf = ceil_rep[n]["recall"], ceil_rep[n]["f1-score"]
    rows.append(dict(cls=n, support=int(ceil_rep[n]["support"]),
                     ceiling_R=round(cr, 3), achieved_R=V3_R[n],
                     ceiling_F1=round(cf, 3), achieved_F1=V3_F1[n],
                     pct_of_ceiling=(round(100 * V3_F1[n] / cf, 1) if cf > 0 else np.nan)))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("\nceiling macro-F1 %.4f  |  achieved macro-F1 0.6329  |  attained %.1f%% of ceiling"
      % (ceil_rep["macro avg"]["f1-score"], 100 * 0.6329 / ceil_rep["macro avg"]["f1-score"]))

           cls  support  ceiling_R  achieved_R  ceiling_F1  achieved_F1  pct_of_ceiling
        Benign    45001      1.000       0.968       1.000        0.983            98.3
      Analysis       77      1.000       0.818       1.000        0.455            45.5
      Backdoor       90      1.000       0.511       1.000        0.634            63.4
           DoS      883      0.999       0.404       0.987        0.462            46.8
      Exploits     6182      0.999       0.696       0.998        0.773            77.4
       Fuzzers     5279      0.992       0.942       0.992        0.708            71.4
       Generic      899      0.970       0.774       0.984        0.805            81.8
Reconnaissance     2406      0.996       0.568       0.998        0.657            65.8
     Shellcode      370      0.895       0.368       0.944        0.367            38.9
         Worms       48      0.958       0.375       0.979        0.486            49.7

ceiling macro-F1 0.9881  |  ach

## 5. Which classes collide?

Counts co-occurrence of each label pair inside identical feature vectors — the label-level structure
of the ambiguity.

In [5]:
cnt = S44["cnt"]
amb = cnt[cnt["k"].isin(S44["amb"])]
piv = amb.pivot_table(index="k", columns="y", values="n", fill_value=0)
for c in range(10):
    if c not in piv.columns:
        piv[c] = 0
piv = piv[sorted(piv.columns)]
A = piv.values

pair = {}
for a, b in combinations(range(10), 2):
    pair[(a, b)] = int(np.minimum(A[:, a], A[:, b]).sum())
top = sorted(pair.items(), key=lambda kv: -kv[1])
print("top colliding label pairs (identical features, different labels):")
for (a, b), v in top[:15]:
    if v:
        print("  %-16s <-> %-16s %6d" % (NAMES[a], NAMES[b], v))

top colliding label pairs (identical features, different labels):
  Fuzzers          <-> Shellcode           134
  Benign           <-> Fuzzers             121
  Fuzzers          <-> Generic              87
  DoS              <-> Fuzzers              47
  Exploits         <-> Fuzzers              46
  DoS              <-> Shellcode            31
  DoS              <-> Reconnaissance       22
  Benign           <-> Shellcode            21
  Exploits         <-> Generic              21
  Benign           <-> Generic              16
  Benign           <-> DoS                  14
  Benign           <-> Exploits             13
  Benign           <-> Reconnaissance       13
  Generic          <-> Shellcode            13
  Reconnaissance   <-> Shellcode            12


## 6. Does the model's confusion mirror the ambiguity?

The decisive test of the whole argument. If the pairs the model confuses are the pairs the *labels*
confuse, the residual error is the data's, not the model's.

The v3 test confusion matrix is transcribed from `two_stage_stageB_ensemble_v3.ipynb` §5.2.

In [6]:
CM = np.array([
 [43546,   51,    2,   18,   73, 1207,  13,   59,  32,  0],
 [    1,   63,    0,    5,    8,    0,   0,    0,   0,  0],
 [    0,    0,   46,    0,   15,   22,   0,    5,   1,  1],
 [    2,   19,    1,  357,  213,  227,  13,   30,  21,  0],
 [    9,   64,    1,  229, 4305, 1213,  58,  246,  52,  5],
 [   32,    2,    2,   18,   74, 4975,  45,   31,  98,  2],
 [    2,    1,    1,   14,   34,  135, 696,    7,   9,  0],
 [    9,    0,    1,   13,  205,  787,   1, 1367,  23,  0],
 [    1,    0,    1,   10,   25,  184,   2,   11, 136,  0],
 [    0,    0,    0,    0,    9,   17,   3,    1,   0, 18]])

model_pair = {(a, b): int(CM[a, b] + CM[b, a]) for a, b in combinations(range(10), 2)}
comp = pd.DataFrame([dict(pair="%s|%s" % (NAMES[a], NAMES[b]),
                          ambiguity=pair[(a, b)], model_confusion=model_pair[(a, b)])
                     for a, b in combinations(range(10), 2)])
rho, p = spearmanr(comp["ambiguity"], comp["model_confusion"])
print("Spearman rho between label ambiguity and model confusion across 45 pairs: %.3f (p=%.2g)" % (rho, p))
print("\ntop 12 by model confusion:")
print(comp.sort_values("model_confusion", ascending=False).head(12).to_string(index=False))
print("\ntop 12 by label ambiguity:")
print(comp.sort_values("ambiguity", ascending=False).head(12).to_string(index=False))

Spearman rho between label ambiguity and model confusion across 45 pairs: 0.699 (p=9e-08)

top 12 by model confusion:
                   pair  ambiguity  model_confusion
       Exploits|Fuzzers         46             1287
         Benign|Fuzzers        121             1239
 Fuzzers|Reconnaissance         10              818
Exploits|Reconnaissance          0              451
           DoS|Exploits          6              442
      Fuzzers|Shellcode        134              282
            DoS|Fuzzers         47              245
        Fuzzers|Generic         87              180
       Exploits|Generic         21               92
        Benign|Exploits         13               82
     Exploits|Shellcode         10               77
      Analysis|Exploits          0               72

top 12 by label ambiguity:
              pair  ambiguity  model_confusion
 Fuzzers|Shellcode        134              282
    Benign|Fuzzers        121             1239
   Fuzzers|Generic         87        

## 7. Findings

**The hypothesis is refuted. Exact label conflict is negligible and is *not* what limits the model.**

| Measure | Value |
|---|---|
| Rows after dedupe | 306,173 |
| **Distinct feature vectors** | **305,602** |
| Ambiguous groups (identical features, ≠ labels) | 497 |
| Rows in ambiguous groups | 1,068 (**0.35%**) |
| Bayes-optimal accuracy | 0.9981 |
| **Irreducible error from exact label conflict** | **0.19%** |
| Ceiling macro-F1 on test | 0.9881 |
| Achieved (v3) | 0.6329 → **64.1% of ceiling** |

The residual 36% is not exact label noise. Four modelling levers failed for some other reason.

### 1. Why this bound is real but weak

305,602 distinct vectors across 306,173 rows — **99.8% of rows are unique**. With continuous flow
features that is unsurprising, and it means the Bayes bound is close to vacuous as a *target*: it is
attained only by a lookup table that memorises unique rows, which generalises to nothing.

§3 demonstrates this directly. A perfect memoriser trained on `train+val` scores **macro-F1 0.0848**
on test, because only **192 test rows (0.31%)** have a feature vector seen in training. It is a
strict upper bound and a useless goal simultaneously.

**So this notebook's contribution is eliminative, not aspirational.** It rules out one explanation for
the plateau; it does not establish that 0.99 is reachable.

One prediction did hold exactly: of those 192 shared vectors, **100% carry a label conflicting with
training**. That follows necessarily from deduplicating on features+label — a vector present in both
splits must have different labels on each side — and it confirms the group construction is behaving
as intended.

### 2. Side result: the feature prune is label-noise-neutral

The 76-feature and 44-feature representations produce **identical** group structure: 305,602 distinct
vectors, 497 ambiguous groups, 1,068 rows, both. The 76→44 prune created **zero** new ambiguity.

That independently validates a decision made much earlier on variance/correlation grounds alone — the
dropped 32 columns carried no information that separated any pair of otherwise-identical flows. Worth
stating in the methodology chapter, since dropping 42% of features usually invites the question.

### 3. But the ambiguity structure *does* predict the model's errors — ρ = 0.699

This is the finding that survives. Across all 45 class pairs, label ambiguity and model confusion
correlate at **Spearman ρ = 0.699 (p = 9×10⁻⁸)**.

| Pair | exact ambiguity | model confusion |
|---|---|---|
| Fuzzers ↔ Shellcode | 134 | 282 |
| Benign ↔ Fuzzers | 121 | 1,239 |
| Fuzzers ↔ Generic | 87 | 180 |
| DoS ↔ Fuzzers | 47 | 245 |
| Exploits ↔ Fuzzers | 46 | **1,287** |

**Fuzzers appears in five of the six largest ambiguity pairs and is simultaneously the model's biggest
error sink** — consistent with the v2 finding that Fuzzers absorbs 3,318 false positives. The 1,068
exactly-colliding rows are too few to matter arithmetically, but they are a *sample* of a much larger
population: pairs of flows that CICFlowMeter cannot distinguish. Where two flows are byte-identical in
44 features yet labelled differently, that is the visible tip of a continuum of near-identical flows.

### 4. The mechanism is feature-space overlap, not label conflict

The exceptions sharpen this. **Exploits ↔ Reconnaissance has 451 model confusions and zero exact
ambiguity**; Analysis ↔ Exploits has 72 confusions and zero. These families are not mislabelled — they
are *inseparable in this representation* while never happening to collide exactly.

That is a different claim from the one this notebook set out to test, and a better-supported one:

> The limit is **class overlap in CICFlowMeter feature space**, not label noise. Attack families in
> CIC-UNSW-NB15-2024 occupy overlapping regions that flow statistics cannot resolve, so every
> classifier, weighting, pool and decision rule inherits the same errors.

This explains all four negative results at once — shared errors across models (DES v1), collapse when
up-weighting hard samples (focal), no gain from moving decision boundaries (thresholds), and
disagreement without complementarity (wider α).

### 5. Per-class: the gap is real, and largest where it hurts

| Class | ceiling F1 | achieved | % of ceiling |
|---|---|---|---|
| Shellcode | 0.944 | 0.367 | **38.9%** |
| Analysis | 1.000 | 0.455 | 45.5% |
| DoS | 0.987 | 0.462 | 46.8% |
| Worms | 0.979 | 0.486 | 49.7% |
| Reconnaissance | 0.998 | 0.657 | 65.8% |
| Fuzzers | 0.992 | 0.708 | 71.4% |
| Exploits | 0.998 | 0.773 | 77.4% |
| Generic | 0.984 | 0.805 | 81.8% |
| Benign | 1.000 | 0.983 | 98.3% |

Shellcode is the only class with meaningful intrinsic ambiguity (ceiling recall 0.895 — 10.5% of its
rows are unrecoverable), and even there the model reaches only 38.9%. **No rare class is anywhere near
its exact-collision ceiling**, so "the labels are noisy" cannot be used to excuse rare-class
performance. The honest framing is overlap, not noise.

### Recommendation for the thesis

Report this as a **methodological control that rules out label noise**, not as a ceiling result. The
usable claims are:

1. Exact label conflict affects 0.35% of rows and cannot explain the performance plateau.
2. The 76→44 prune introduces no label ambiguity — an independent validation of the feature selection.
3. Label-collision structure predicts model confusion at ρ = 0.70, localising the difficulty to
   specific family pairs (Fuzzers-centred) rather than to the classifier.

### Next step

The overlap hypothesis now needs a **non-vacuous measure**, which exact matching cannot provide. The
natural instrument is a **k-NN label-overlap analysis** in RobustScaler space: for each attack row,
the label composition of its k nearest neighbours. That yields a per-class *soft* purity score and a
generalisation-aware overlap estimate, and — unlike this bound — it will produce a ceiling that is
actually informative about achievable performance.
